# 6.2 参数高效微调 (PEFT)

参数高效微调仅训练少量参数即可将预训练知识迁移到目标任务，大幅降低计算成本。

本节涵盖：LoRA, QLoRA, Adapter, Prefix Tuning, Prompt Tuning, P-Tuning v2, IA³, DoRA> 🕐 预估学习时间：40分钟

## 1. LoRA

**目的**：以极低参数量实现接近全参数微调的效果

**基本原理**：在预训练权重矩阵旁添加低秩分解矩阵ΔW=BA（B∈R^(d×r), A∈R^(r×d)，r远小于d），仅训练A和B，推理时将ΔW合并回原权重，无额外推理开销。

**核心公式**：
- h = Wx + ΔWx = Wx + BAx
- 缩放因子：α/r，α控制更新幅度
- 推理时合并：W_new = W + α/r × BA

**关键优势**：
- 参数量仅占原模型的0.1-1%
- 推理时无额外开销（合并回原权重）
- 可为不同任务训练不同的LoRA，灵活切换

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank=8, alpha=16):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.W.weight.requires_grad = False
        self.A = nn.Parameter(torch.randn(rank, in_dim) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_dim, rank))
        self.scaling = alpha / rank

    def forward(self, x):
        return self.W(x) + self.scaling * (x @ self.A.T @ self.B.T)

d_model = 1024
rank = 8
lora = LoRALayer(d_model, d_model, rank=rank)

full_params = d_model * d_model
lora_params = rank * d_model * 2

x = torch.randn(2, 16, d_model)
out = lora(x)

print('=== LoRA (Low-Rank Adaptation) ===')
print(f'Full linear layer: {full_params:,} params')
print(f'LoRA (rank={rank}): {lora_params:,} params ({lora_params/full_params*100:.2f}%)')
print(f'Output shape: {out.shape}')

with torch.no_grad():
    merged_weight = lora.W.weight + lora.scaling * (lora.B @ lora.A)
print(f'\nMerged weight for inference: {merged_weight.shape} (no extra latency)')

print(f'\nKey: LoRA adds only {lora_params/full_params*100:.2f}% parameters')
print(f'but achieves performance close to full fine-tuning.')

## 2. QLoRA

**目的**：在LoRA基础上进一步降低显存需求

**基本原理**：将预训练模型量化为4-bit（NF4格式），在量化模型上添加LoRA适配器进行训练，使用分页优化器管理显存峰值。

**NF4量化**：NormalFloat4，专为正态分布权重设计的4-bit格式
- 假设权重服从正态分布
- 将量化级别均匀分布在正态分布的分位数上
- 比均匀量化更精确

**代表成果**：单张A100 80GB可微调65B模型

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

def nf4_quantize(tensor):
    shape = tensor.shape
    flat = tensor.flatten()
    abs_max = flat.abs().max()
    normalized = flat / abs_max
    n_levels = 16
    quantile_values = torch.tensor([(-1 + 2*i/(n_levels-1)) for i in range(n_levels)])
    indices = torch.searchsorted(quantile_values, normalized)
    indices = indices.clamp(0, n_levels - 1)
    dequantized = quantile_values[indices] * abs_max
    return dequantized.reshape(shape), abs_max

W = torch.randn(512, 512)
W_nf4, scale = nf4_quantize(W)

error = (W - W_nf4).abs().mean()
relative_error = error / W.abs().mean()

print('=== QLoRA (4-bit Quantized LoRA) ===')
print(f'Original: FP32, {W.numel() * 4 / 1e6:.2f} MB')
print(f'NF4: 4-bit, {W.numel() * 0.5 / 1e6:.2f} MB (8x compression)')
print(f'Quantization error: {relative_error:.4f} ({relative_error:.2%})')

model_size_b = 7
fp16_mem = model_size_b * 2
nf4_mem = model_size_b * 0.5
lora_mem = 0.01 * model_size_b * 2
print(f'\n7B model memory comparison:')
print(f'  FP16 base model: {fp16_mem:.1f} GB')
print(f'  NF4 base model: {nf4_mem:.1f} GB')
print(f'  LoRA adapters: ~{lora_mem:.2f} GB')
print(f'  QLoRA total: ~{nf4_mem + lora_mem:.1f} GB')
print(f'\nKey: QLoRA enables fine-tuning large models on consumer GPUs.')

## 3. Adapter / Prefix Tuning / Prompt Tuning / P-Tuning v2 / IA³ / DoRA

**Adapter**：在Transformer层中插入下投影-非线性-上投影的瓶颈结构，仅训练Adapter参数，但会增加推理延迟。

**Prefix Tuning**：在每层注意力的K和V前拼接可学习的前缀向量（virtual tokens），仅训练前缀参数。

**Prompt Tuning**：仅在输入嵌入层前添加可学习的连续向量（soft prompts），参数量极少。

**P-Tuning v2**：在Transformer的每层都添加可学习的深度提示，使中小模型也能获得与全参数微调相当的效果。

**IA³**：通过学习三个向量分别对注意力中的K、V和FFN中的激活进行逐元素缩放，参数量比LoRA更少。

**DoRA**：将LoRA的权重更新分解为幅度（magnitude）和方向（direction）两个分量分别学习，更接近全参数微调的权重更新模式。

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class AdapterLayer(nn.Module):
    def __init__(self, d_model, bottleneck=32):
        super().__init__()
        self.down = nn.Linear(d_model, bottleneck)
        self.up = nn.Linear(bottleneck, d_model)
        self.act = nn.ReLU()

    def forward(self, x):
        return x + self.up(self.act(self.down(x)))

class IA3Layer(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.scale_k = nn.Parameter(torch.ones(n_heads, 1))
        self.scale_v = nn.Parameter(torch.ones(n_heads, 1))
        self.scale_ffn = nn.Parameter(torch.ones(d_model))

class DoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank=8, alpha=16):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.W.weight.requires_grad = False
        self.lora_A = nn.Parameter(torch.randn(rank, in_dim) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_dim, rank))
        self.scaling = alpha / rank
        with torch.no_grad():
            self.magnitude = nn.Parameter(self.W.weight.norm(dim=1))

    def forward(self, x):
        base_out = self.W(x)
        lora_out = self.scaling * (x @ self.lora_A.T @ self.lora_B.T)
        combined = base_out + lora_out
        norm = combined.norm(dim=-1, keepdim=True)
        return self.magnitude.unsqueeze(0) * combined / (norm + 1e-8)

d_model = 512
methods = {
    'LoRA': d_model * 8 * 2,
    'Adapter': d_model * 32 * 2,
    'Prefix Tuning': 10 * d_model,
    'Prompt Tuning': 10 * d_model,
    'IA3': d_model + 8 + 8,
    'DoRA': d_model * 8 * 2 + d_model,
}

full = d_model * d_model
print('=== PEFT Methods Comparison ===')
print(f'Full linear: {full:,} params\n')
for name, params in methods.items():
    print(f'{name:16s}: {params:>8,} params ({params/full*100:.3f}%)')

adapter = AdapterLayer(d_model)
dora = DoRALayer(d_model, d_model)
x = torch.randn(2, 16, d_model)
print(f'\nAdapter output: {adapter(x).shape}')
print(f'DoRA output: {dora(x).shape}')
print(f'\nKey: All PEFT methods train <1% of parameters.')
print(f'LoRA/DoRA: most popular, no inference overhead after merging.')
print(f'Adapter: adds inference latency due to extra layers.')

## 📝 课后思考题1. **LoRA 的 rank 选择如何影响微调效果？过大和过小的 rank 各有什么问题？**   <details><summary>💡 点击查看提示</summary>      **参考答案**：   - **rank 过小（如 r=1, 2）**：     - 优点：参数量极少，训练快，显存占用最小。     - 问题：低秩空间的表达能力不足，无法捕获任务所需的全部权重更新方向，性能显著低于全参数微调。     - 适用场景：极简单任务、资源极度受限。   - **rank 过大（如 r=256, 512）**：     - 优点：低秩近似更精确，理论性能接近全参数微调。     - 问题：参数量增加，训练变慢，可能出现过拟合（尤其在小数据集上），且收益递减。   - **最佳实践**：     - 常用 rank=8, 16, 32 在大多数任务中取得良好平衡。     - r=64~128 用于复杂任务或需要精确知识注入的场景。     - 经验法则：任务越复杂/领域差异越大，需要越大的rank。      </details>2. **QLoRA 的 4-bit NormalFloat 量化与标准 INT4 量化有什么区别？**   <details><summary>💡 点击查看提示</summary>      **参考答案**：   - **INT4 均匀量化**：     - 将值域[-max, max]均匀划分为16个区间。     - 量化级别等间距分布。     - 简单高效，但对正态分布权重利用率不高——大部分值集中在0附近，两端区间几乎空闲。   - **NF4（NormalFloat4）量化**：     - 假设权重服从N(0,1)正态分布。     - 量化级别均匀分布在正态分布的分位数上（而非值域上）。     - 0附近密集分配更多量化级别，两端稀疏分配。     - 对Transformer权重的正态分布特性天然适配。   - **核心区别**：NF4在每个量化区间期望包含相同数量的值，最大化信息保留；INT4在值域空间均匀，可能浪费量化精度。   - **实际效果**：NF4在同等4-bit精度下比INT4保留更多信息，QLoRA的微调效果更接近全精度。      </details>3. **在什么场景下全参数微调仍然优于 PEFT 方法？**   <details><summary>💡 点击查看提示</summary>      **参考答案**：   - **领域差异极大时**：当目标领域与预训练语料差异巨大（如将通用LLM微调为专业医学/法律模型），低秩更新可能不足以覆盖所需的权重变化幅度。   - **数据量充足时**：当拥有大量高质量标注数据（如数百万条），全参数微调可以充分挖掘数据价值，通常能稳定优于PEFT。   - **需要学习新知识而非格式适配时**：PEFT更擅长格式适配和风格迁移；如果模型缺乏目标领域的核心知识（如新语言、新编程语言），全参数微调效果更好。   - **追求极致性能时**：在benchmark上通常全参数微调仍略优于LoRA等PEFT方法，如果条件允许且追求SOTA，全参数微调是更优选择。   - **需要改变模型核心行为时**：如改变模型的价值观、安全对齐层等，全参数微调的影响范围更大。   - **判断标准**：用小规模实验对比全参数vs LoRA，如果差距>5%，且资源充足，选全参数；否则PEFT性价比更高。      </details>